
# Business Challenge

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

(Thanks to Edward Donner for the lesson!)

---
## Setup

In [1]:
# Imports

import json
from IPython.display import Markdown, display, update_display

from openai import OpenAI
MODEL = "llama3.2"

from bs4 import BeautifulSoup
import requests

In [ ]:
# Connect to the Ollama API
llama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

---

## Part 1 - Relevant Links' Extraction

### Scraper and Link Extractor

In [3]:
# Website Scrapers 

# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]


def fetch_website_links(url):
    """
    Return the links on the webiste at the given url
    I realize this is inefficient as we're parsing twice! This is to keep the code in the lab simple.
    Feel free to use a class and optimize it!
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]

### Define Prompts and Prompt Generators

In [12]:
# system prompt for the links' extracton task
link_system_prompt = """
    You are provided with a list of links found on a webpage.
    You are to decide which of the links would be most relevant to include in a brochure about the company,
    such as links to an About page, or a Company page, or Careers/Jobs pages.
    You should respond in JSON as in this example:

    {
        "links": [
            {"type": "about page", "url": "https://full.url/goes/here/about"},
            {"type": "careers page", "url": "https://another.full.url/careers"}
        ]
    }
"""

In [13]:
# user prompt for the links' extraction task

def get_links_user_prompt(url):
    user_prompt = f"""
        Here is the list of links on the website {url} -
        Please decide which of these are relevant web links for a brochure about the company, 
        respond with the full https URL in JSON format.
        Do not include Terms of Service, Privacy, email links.

        Links (some might be relative links):

    """
    
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [14]:
# run the above function with input to check out the resulting user prompt
print(get_links_user_prompt("https://www.apple.com/"))


        Here is the list of links on the website https://www.apple.com/ -
        Please decide which of these are relevant web links for a brochure about the company, 
        respond with the full https URL in JSON format.
        Do not include Terms of Service, Privacy, email links.

        Links (some might be relative links):

    /
/us/shop/goto/store
/mac/
/ipad/
/iphone/
/watch/
/apple-vision-pro/
/airpods/
/tv-home/
/entertainment/
/us/shop/goto/buy_accessories
https://support.apple.com/?cid=gn-ols-home-hp-tab
/us/search
/us/shop/goto/bag
/iphone-18-pro/
/iphone-18-pro/
/us/shop/goto/buy_iphone/iphone_18_pro
/iphone-duo/
/iphone-duo/
/us/shop/goto/buy_iphone/iphone_duo
/apple-watch-series-12/
/apple-watch-series-12/
/us/shop/goto/buy_watch/apple_watch_series_12
/apple-watch-ultra-4/
/apple-watch-ultra-4/
/us/shop/goto/buy_watch/apple_watch_ultra_4
/macbook-air/
/macbook-air/
/us/shop/goto/buy_mac/macbook_air
/ipad-air/
/ipad-air/
/us/shop/goto/buy_ipad/ipad_air
/airpods-pro

### Link Extraction with Llama3.2

In [18]:
# link extraction function that uses the above system and user prompts to extract relevant links from a website
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")

    response = llama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    
    result = response.choices[0].message.content # JSON string from the model
    links = json.loads(result) # JSON string to python dictionary

    print(f"Found {len(links['links'])} relevant links")
    print(links)
    return links

In [11]:
select_relevant_links("https://www.apple.com")

Selecting relevant links for https://www.apple.com by calling llama3.2
Found 35 relevant links


{'links': [{'type': 'Company page', 'url': 'https://www.apple.com'},
  {'type': 'About page',
   'url': 'https://www.apple.com/investor relations/default.aspx'},
  {'type': 'Careers page', 'url': 'https://careers.apple.com/'},
  {'type': 'Newsroom', 'url': 'https://www.apple.com/newsroom/'},
  {'type': 'Investor Relations', 'url': 'https://investor.apple.com/'},
  {'type': 'Terms of Service', 'url': 'https://www.apple.com/terms.html'},
  {'type': 'FAQs', 'url': 'https://www.apple.com/FAQ/'},
  {'type': 'Contact', 'url': 'https://www.apple.com/contact/'},
  {'type': 'Support', 'url': 'https://support.apple.com/'},
  {'type': 'Apple Card', 'url': 'https://card.apple.com/apply/application'},
  {'type': 'Apple Wallet', 'url': 'https://applepay.apple.com/'},
  {'type': 'Apple Cash',
   'url': 'https://apple-cash.apple.com/redirect?uri=https://apple-cash.apple.com/'},
  {'type': 'Apple Pay', 'url': 'https://applepay.apple.com/'},
  {'type': 'Services', 'url': 'https://www.apple.com/services/

--- 
## Part 2 - Create the Brochure

In [31]:
# function to fetch the main page content and all relevant links from a website
def fetch_content_of_page_and_for_all_relevant_links(url):
    
    # fetch the main page content
    contents = fetch_website_contents(url)
    
    # fetch the relevant links with the LLM
    relevant_links = select_relevant_links(url)
    
    # format the result as a markdown string
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"

    # fetch the contents of each relevant link and append to the result
    for link in relevant_links['links']:

        try:
            # fetch the contents of the relevant link
            link_contents = fetch_website_contents(link["url"])

            result += f"\n\n### Link: {link['type']}\n"
            result += link_contents
        except Exception as e:
            print(f"Error fetching contents for {link['url']}: {e}")
            # link_contents = "Error fetching contents."


    return result

In [32]:
# run the content fetching function for a website and display the result as markdown
result = fetch_content_of_page_and_for_all_relevant_links("https://www.apple.com")

print(result)

Selecting relevant links for https://www.apple.com by calling llama3.2
Found 35 relevant links
{'links': [{'type': 'home page', 'url': 'https://www.apple.com/'}, {'type': 'about.page', 'url': 'https://account.apple.com/'}, {'type': 'investor relations page', 'url': 'https://investor.apple.com/'}, {'type': 'contact us page', 'url': 'https://contact.apple.com/'}, {'type': 'careers page', 'url': 'https://careers.apple.com/'}, {'type': 'leadership page', 'url': 'https://newsroom.apple.com/about-leadership'}, {'type': 'newsroom page', 'url': 'https://newsroom.apple.com/'}, {'type': 'supply chain page', 'url': 'https://www.apple.com/supply-chains/'}, {'type': 'government page', 'url': 'https://www.apple.com/government/'}, {'type': 'education page', 'url': 'https://www.apple.com/education/'}, {'type': 'accessibility page', 'url': 'https://www.apple.com/accessibility/'}, {'type': 'environment page', 'url': 'https://www.apple.com/environment/'}, {'type': 'diversity page', 'url': 'https://www.ap

### Define Prompts and Prompt Generator

In [33]:
# system prompt for brochure generation task

brochure_system_prompt = """
    You are an assistant that analyzes the contents of several relevant pages from a company website
    and creates a short brochure about the company for prospective customers, investors and recruits.
    Respond in markdown without code blocks.
    Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
#   You are an assistant that analyzes the contents of several relevant pages from a company website
#   and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
#   Respond in markdown without code blocks.
#   Include details of company culture, customers and careers/jobs if you have the information.
#   """

In [34]:
# user prompt generator function
def get_brochure_user_prompt(company_name, url):
    
    user_prompt = f"""
    You are looking at a company called: {company_name}
    Here are the contents of its landing page and other relevant pages;
    use this information to build a short brochure of the company in markdown without code blocks.\n\n
    """

    user_prompt += fetch_content_of_page_and_for_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    
    return user_prompt

In [35]:
# Call the brochure generation user prompt function
get_brochure_user_prompt("Apple", "https://www.apple.com")

Selecting relevant links for https://www.apple.com by calling llama3.2
Found 14 relevant links
{'links': [{'type': 'About us', 'url': 'https://www.apple.com/'}, {'type': 'Apple Card', 'url': 'https://apple-card.apple.com/apply/application?referrer=cid%3Dapy-200-10000036&start=false'}, {'type': 'Apple Upgrade', 'url': 'https://appleUpgrade.apple.com/'}, {'type': 'Apple Events', 'url': 'https://www.apple.com/apple-events/'}, {'type': 'Investor Relations', 'url': 'https://investor.apple.com/'}, {'type': 'Careers', 'url': 'https://careers.apple.com/us/'}, {'type': 'Newsroom', 'url': 'https://news.apple.com/'}, {'type': 'Sitemap', 'url': 'https://www.apple.com/sitemap/'}, {'type': 'Contact', 'url': 'https://contact.apple.com/'}, {'type': 'Legal', 'url': 'https://www.apple.com/legal/'}, {'type': 'Retail', 'url': 'https://www.apple.com/retail/'}, {'type': 'Retail Geniusbar', 'url': 'https://www.apple.com/retail/geniusbar/'}, {'type': 'Support', 'url': 'https://support.apple.com/'}, {'type': '

'\n    You are looking at a company called: Apple\n    Here are the contents of its landing page and other relevant pages;\n    use this information to build a short brochure of the company in markdown without code blocks.\n\n\n    ## Landing Page:\n\nApple\n\nApple\nApple\nStore\nMac\niPad\niPhone\nWatch\nVision\nAirPods\nTV & Home\nEntertainment\nAccessories\nSupport\n0\n+\niPhone 18 Pro\nPro further.\nLearn more\nBuy\niPhone Duo\nHello, hello.\nPre-order starting 5:00 a.m. PT on 10.16 Available\xa0starting\xa010.23\nLearn more\nView pricing\nApple Watch Series 12\nThe most accurate heart rate sensing in a wearable.\n1\nLearn more\nBuy\nApple Watch Ultra 4\nA battery you can’t outrun.\nLearn more\nBuy\nMacBook\xa0Air\nNow supercharged by M5.\nLearn more\nBuy\niPad Air\nNow supercharged by M4.\nLearn more\nBuy\nAirPods Pro 3\nThe world’s best in-ear Active Noise Cancellation.\nLearn more\nBuy\nApple Upgrade\nLove it. Lease it. Upgrade it.\n2\nLearn more\nApple Card\nGet up to 3% Daily

### Generate brochure

In [39]:
# brochure generation function that uses the above system and user prompts to generate a brochure for a company
def generate_brochure(company_name, url):
    response = llama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    
    result = response.choices[0].message.content
    
    display(Markdown(result))

In [40]:
# generate the brochure for Apple
generate_brochure("Apple", "https://www.apple.com")

Selecting relevant links for https://www.apple.com by calling llama3.2
Found 4 relevant links
{'links': [{'type': 'company page', 'url': 'https://www.apple.com/'}, {'type': 'About page', 'url': 'https://about.apple.com/'}, {'type': 'Careers page', 'url': 'https://careers.apple.com/'}, {'type': 'apple card', 'url': 'https://card.apple.com/'}]}
Error fetching contents for https://about.apple.com/: HTTPSConnectionPool(host='about.apple.com', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='about.apple.com', port=443): Failed to resolve 'about.apple.com' ([Errno -5] No address associated with hostname)"))


# Welcome to Apple

At Apple, we're passionate about creating innovative technology that empowers people to change the world. We're a company of more than 170,000 employees, and we're committed to making a positive impact on our customers, our communities, and the environment.

## Our Mission

Our mission is to bring the best user experience to our customers through our premium products and services. We believe that technology should be simple, easy to use, and accessible to everyone. Our goal is to create products that make a dent in the universe and inspire people to do something they haven't done before.

## Company Culture

Our company culture is built around our values of innovation, creativity, and teamwork. We encourage our employees to think outside the box, take risks, and push the boundaries of what's possible. We also prioritize diversity, equity, and inclusion, and we're committed to creating a workplace where everyone feels welcome and valued.

## Our Customers

We're proud to serve some of the world's most innovative and creative customers, from artists and musicians to entrepreneurs and inventors. We also work closely with education institutions, governments, and non-profit organizations to create technology that makes a positive impact on society.

## Careers

If you're passionate about technology and want to join a team that's changing the world, we've got a career that's right for you. Check out our careers page to learn more about our open positions and how to apply.

## Our Products and Services

We offer a range of innovative products and services, from iPhones and Macs to iPads and Apple Watches. We also offer Apple Upgrade, a device leasing program that makes it easy to upgrade your technology.

## What We're Good At

We're good at:

* Creating technology that is simple, easy to use, and accessible to everyone
* Delivering excellent customer service
* Innovating and pushing the boundaries of what's possible
* Making a positive impact on our customers, our communities, and the environment

## Get in Touch

If you have questions or just want to learn more about Apple, we'd love to hear from you. Check out our contact page to get in touch with us.

# Join the Team

We're looking for talented individuals who share our values and passion for technology. Check out our careers page to learn more about our open positions and how to apply.

---

## Part 3 - Stream the Brochure

**Stream the result so it reads like the familiar typewriter animation in AI chat solutions**

In [41]:
# define streaming brochure generation function that uses the above system and user prompts to generate a brochure for a company
def stream_brochure(company_name, url):
    stream = llama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )

    response = ""
    
    display_handle = display(Markdown(""), display_id=True)
    
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [42]:
# stream the brochure for Apple
stream_brochure("Apple", "https://www.apple.com")

Selecting relevant links for https://www.apple.com by calling llama3.2
Found 17 relevant links
{'links': [{'type': 'about page', 'url': 'https://about.apple.com/'}, {'type': ' careers page', 'url': 'https://careers.apple.com/us/'}, {'type': 'newsroom page', 'url': 'https://news.apple.com/'}, {'type': 'leadership page', 'url': 'https://leadership.apple.com/'}, {'type': 'investor relations page', 'url': 'https://investor.apple.com/'}, {'type': 'store page', 'url': 'https://www.apple.com'}, {'type': 'support page', 'url': 'https://support.apple.com/'}, {'type': 'account page', 'url': 'https://account.apple.com/'}, {'type': 'icloud page', 'url': 'https://www.icloud.com'}, {'type': 'apple watch series 12 page', 'url': 'https://apple.com/watch-series-12/'}, {'type': 'apple watch ultra 4 page', 'url': 'https://apple.com/apple-watch-ultra-4/'}, {'type': 'macbook air page', 'url': 'https://apple.com/macbook-air/'}, {'type': 'ipad air page', 'url': 'https://apple.com/ipad-air/'}, {'type': 'airpo

# Apple Brochure

## About Us

Apple is a technology company that designs, manufactures, and markets consumer electronics, computer software, and online services. With a focus on innovation and customer satisfaction, Apple aims to create products that are not only functional but also beautiful and easy to use.

## Our Products

Apple's product lineup is extensive, ranging from iPhones and iPads to Macs, Apple Watches, and AirPods. Each product is designed to integrate seamlessly with other Apple devices, allowing customers to seamlessly switch between their different Apple products. Apple also offers a range of accessories, including Apple Watch bands, cases, and headphones.

## Our Values

At Apple, our values are centered around innovation, customer satisfaction, and sustainability. We believe in pushing the boundaries of what technology can do, and we're committed to creating products that are both functional and beautiful. We also believe in giving back to our communities, and we're dedicated to making a positive impact on the world around us.

## Company Culture

Apple is known for its unique company culture, which is built around the idea of creating a workplace that is both fun and productive. Our team is made up of talented individuals who share a passion for technology and innovation, and we strive to create a work environment that is collaborative, inclusive, and empowering.

## Customers

Millions of customers around the world rely on Apple products to power their personal and professional lives. From individuals who use our products to enhance their creativity and productivity to businesses that use our products to stay ahead of the curve, Apple products are designed to meet the needs of a diverse range of customers.

## Careers

Apple is committed to providing talented individuals with opportunities to join our team and contribute to our mission of accelerating technological innovation. We're hiring talented individuals across a range of roles, from software engineers to sales representatives, and from customer support specialists to data analysts.

Whether you're a tech enthusiast, a creative professional or a business leader, there is a career opportunity at Apple that's right for you.

## Customer Reviews

Apple has received widespread critical acclaim and admiration from customers around the world. Many have praised the company's commitment to innovation and customer satisfaction, while others have appreciated the sleek and user-friendly design of Apple products.

"We love our iPhones! They're the best phone we've ever had." - Anonymous Apple Customer

"I was skeptical about using an Apple Watch, but now I couldn't imagine going without it." - Rachel, San Francisco

"I've never seen such a seamless and intuitive experience in a software product. It's changed the way I work." - Anonymous Business Owner

## Join the Team

Join Apple to be part of a team that's shaping the future of technology and innovation. From software engineers to sales representatives, and from customer support specialists to data analysts, we have a range of roles available across our different business units.

Ready to join the Apple team? Visit our careers page to explore current job openings and learn more about our company culture and vision.